In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

vendor_name = "entity_resolution"
layer_name = "silver_entity_resolution"

print("--- Starting Silver Layer Stage 2: Entity Resolution ---")


def normalize_vendor(
    df,
    *,
    site_id,
    street_address,
    city,
    state,
    zip_code,
    lat,
    lon,
    network_type=None,
    status=None,
    source_vendor,
):
    def optional_col(name, dtype="string"):
        return F.col(name).cast(dtype) if name else F.lit(None).cast(dtype)

    return df.select(
        F.col(site_id).cast("string").alias("site_id"),
        optional_col(street_address).alias("street_address"),
        optional_col(city).alias("city"),
        optional_col(state).alias("state"),
        optional_col(zip_code).alias("zip"),
        F.col(lat).cast("double").alias("lat"),
        F.col(lon).cast("double").alias("lon"),
        optional_col(network_type).alias("network_type"),
        optional_col(status).alias("status"),
        F.lit(source_vendor).alias("source_vendor"),
    )


# 1. Read all non-registry silver tables and map them into a common schema
df_here = normalize_vendor(
    spark.table("inlap.silver.heretech_conformed"),
    site_id="site_id",
    street_address="street_address",
    city="city",
    state="state",
    zip_code="zip",
    lat="latitude",
    lon="longitude",
    network_type="network_type",
    status=None,
    source_vendor="here_tech",
)

df_open = normalize_vendor(
    spark.table("inlap.silver.opensignal_conformed"),
    site_id="site_ref",
    street_address=None,
    city=None,
    state=None,
    zip_code=None,
    lat="latitude",
    lon="longitude",
    network_type="network_type",
    status=None,
    source_vendor="opensignal",
)

df_geo = normalize_vendor(
    spark.table("inlap.silver.georesults_conformed"),
    site_id="site_id",
    street_address="address1",
    city="municipality",
    state="state",
    zip_code="postal_code",
    lat="lat",
    lon="lon",
    network_type=None,
    status=None,
    source_vendor="georesults",
)

df_prec = normalize_vendor(
    spark.table("inlap.silver.precisely_conformed"),
    site_id="LOC_ID",
    street_address="ADDR_FULL",
    city="city",
    state="state_cd",
    zip_code="zip",
    lat="latitude",
    lon="longitude",
    network_type=None,
    status="VALIDATION_STATUS",
    source_vendor="precisely",
)

df_nv5 = normalize_vendor(
    spark.table("inlap.silver.nv5_conformed"),
    site_id="SITEID",
    street_address="addr",
    city="cty",
    state="st",
    zip_code="ZIP",
    lat="latitude",
    lon="longitude",
    network_type=None,
    status="normalized_status",
    source_vendor="nv5",
)

df_source_records = (
    df_here
    .unionByName(df_open)
    .unionByName(df_geo)
    .unionByName(df_prec)
    .unionByName(df_nv5)
    .withColumn("lat_rounded", F.round(F.col("lat"), 4))
    .withColumn("lon_rounded", F.round(F.col("lon"), 4))
)

# 2. Build a coordinate-to-GLID rollup from the Geolink registry.
#    Business rule: when a coordinate resolves to multiple GLIDs in the same building,
#    keep the coordinate/baseGLID grain instead of choosing an arbitrary apartment or suite GLID.
df_geolink_registry = spark.table("inlap.silver.geolink_conformed").select(
    F.col("baseglid"),
    F.col("glid"),
    F.col("lat_rounded"),
    F.col("lon_rounded"),
    F.col("full_address").alias("resolved_street_address"),
    F.col("unit_number").alias("resolved_unit_number"),
    F.col("city").alias("resolved_city"),
    F.col("state").alias("resolved_state"),
    F.col("zip").alias("resolved_zip"),
    F.col("location_type").alias("resolved_location_type"),
    F.col("confidence_score").alias("registry_confidence_score"),
    F.col("last_verified").alias("registry_last_verified"),
)

coord_window = Window.partitionBy("lat_rounded", "lon_rounded").orderBy(
    F.when(F.col("glid") == F.col("baseglid"), F.lit(0)).otherwise(F.lit(1)).asc(),
    F.when(
        F.col("resolved_unit_number").isNull() | (F.trim(F.col("resolved_unit_number")) == ""),
        F.lit(0),
    ).otherwise(F.lit(1)).asc(),
    F.col("registry_confidence_score").desc(),
    F.col("registry_last_verified").desc(),
    F.col("glid").asc(),
)

df_geolink_representative = (
    df_geolink_registry
    .withColumn("coord_rank", F.row_number().over(coord_window))
    .filter(F.col("coord_rank") == 1)
    .drop("coord_rank")
    .withColumnRenamed("baseglid", "representative_baseglid")
    .withColumnRenamed("glid", "representative_glid")
)

df_geolink_coordinate_summary = (
    df_geolink_registry.groupBy("lat_rounded", "lon_rounded")
    .agg(
        F.countDistinct("baseglid").alias("matched_baseglid_count"),
        F.countDistinct("glid").alias("matched_glid_count"),
        F.max("registry_last_verified").alias("latest_registry_verification"),
        F.max("registry_confidence_score").alias("max_registry_confidence"),
    )
)

df_geolink_coordinate_rollup = df_geolink_coordinate_summary.join(
    df_geolink_representative,
    ["lat_rounded", "lon_rounded"],
    "left",
)

# 3. Resolve every source record against the registry coordinate rollup.
df_enriched = (
    df_source_records.join(df_geolink_coordinate_rollup, ["lat_rounded", "lon_rounded"], "left")
    .withColumn(
        "geolink_match_status",
        F.when(F.col("matched_glid_count").isNull(), F.lit("no_match"))
        .when(F.col("matched_glid_count") == 1, F.lit("single_glid"))
        .when(F.col("matched_baseglid_count") == 1, F.lit("ambiguous_multi_unit"))
        .otherwise(F.lit("ambiguous_multi_location")),
    )
    .withColumn(
        "resolved_baseglid",
        F.when(F.col("matched_baseglid_count") == 1, F.col("representative_baseglid")),
    )
    .withColumn(
        "resolved_glid",
        F.when(F.col("matched_glid_count") == 1, F.col("representative_glid"))
        .when(F.col("matched_baseglid_count") == 1, F.col("representative_baseglid")),
    )
    .withColumn(
        "registry_join_grain",
        F.when(F.col("geolink_match_status") == "single_glid", F.lit("coordinate_glid"))
        .when(F.col("geolink_match_status") == "ambiguous_multi_unit", F.lit("coordinate_baseglid"))
        .otherwise(F.lit("coordinate_only")),
    )
    .withColumn("canonical_street_address", F.coalesce(F.col("street_address"), F.col("resolved_street_address")))
    .withColumn("canonical_city", F.coalesce(F.col("city"), F.col("resolved_city")))
    .withColumn("canonical_state", F.coalesce(F.col("state"), F.col("resolved_state")))
    .withColumn("canonical_zip", F.coalesce(F.col("zip"), F.col("resolved_zip")))
)

fallback_master_id = F.sha2(
    F.concat_ws(
        "|",
        F.col("lat_rounded").cast("string"),
        F.col("lon_rounded").cast("string"),
        F.coalesce(F.col("canonical_state"), F.lit("UNK")),
        F.coalesce(F.col("canonical_zip"), F.lit("UNK")),
    ),
    256,
)

df_master_assigned = df_enriched.withColumn(
    "master_site_id",
    F.coalesce(F.col("site_id"), F.col("resolved_glid"), fallback_master_id),
)

# Bronze enforcement: only rows with non-null required fields are written to silver
# (site_id, lat, lon, source_vendor must all be present)
df_silver_ready = df_master_assigned.filter(
    F.col("site_id").isNotNull()
    & F.col("lat").isNotNull()
    & F.col("lon").isNotNull()
    & F.col("source_vendor").isNotNull()
)

# 4. Refresh the unified site table as external Delta under ADLS silver folder
SILVER_BASE = "abfss://datalake@attinlapsa.dfs.core.windows.net/silver"

df_silver_ready.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{SILVER_BASE}/sites_unified/")

match_summary = {
    row["geolink_match_status"]: row["count"]
    for row in df_silver_ready.groupBy("geolink_match_status").count().collect()
}

print(
    f"Entity resolution complete. Unified master table updated with {df_silver_ready.count()} records. "
    f"Geolink match summary: {match_summary}"
)

display(
    df_silver_ready
    .orderBy(F.col("source_vendor").asc(), F.col("site_id").asc())
    .limit(5)
)